# R3 · Data Warehouse, integración y OLAP

**Responsable:** Nicolás (Rol A)  
**Revisor:** Cristian (Rol C)

## Qué queda resuelto

- [x] Modelo en estrella en DuckDB (`src/02_construir_dw.py`) y conciliación contra Parquet
- [x] Integración de OWID y Banco Mundial por ISO3 con tasa de cruce
- [x] Consultas OLAP: roll-up, drill-down, slice y dice sobre el DW

In [ ]:
import sys, pathlib
RAIZ = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(RAIZ))
import duckdb
import pandas as pd

DW_PATH = str(RAIZ / 'data' / 'dw_covid.duckdb')
PARQUET_PATH = str(RAIZ / 'data' / 'clean' / 'reportes_unificados.parquet')
con = duckdb.connect(DW_PATH, read_only=True)

## Conciliación estrella en DuckDB vs. Parquet

In [ ]:
filas_pq = con.execute(f"SELECT COUNT(*) FROM read_parquet('{PARQUET_PATH}')").fetchone()[0]
filas_fact = con.execute('SELECT COUNT(*) FROM fact_covid').fetchone()[0]
print(f'Filas Parquet: {filas_pq:,} | Filas fact_covid: {filas_fact:,} | Duplicados consolidados: {filas_pq - filas_fact:,}')
display(con.execute('SHOW TABLES').df())

## Integración de fuentes externas (OWID y Banco Mundial) por ISO3 y tasa de cruce

In [ ]:
cruce = con.execute('''
    SELECT
        COUNT(DISTINCT t.country_region) AS paises_dw,
        COUNT(DISTINCT CASE WHEN v.iso3 IS NOT NULL THEN t.country_region END) AS cruzan_owid,
        COUNT(DISTINCT CASE WHEN w.iso3 IS NOT NULL THEN t.country_region END) AS cruzan_wb
    FROM dim_territorio t
    LEFT JOIN (SELECT DISTINCT iso3 FROM fact_vacunacion) v ON t.iso3 = v.iso3
    LEFT JOIN dim_indicadores_wb w ON t.iso3 = w.iso3
''').df()
cruce['tasa_owid_pct'] = (cruce['cruzan_owid'] / cruce['paises_dw'] * 100).round(1)
cruce['tasa_wb_pct'] = (cruce['cruzan_wb'] / cruce['paises_dw'] * 100).round(1)
display(cruce)

## Consultas OLAP: roll-up, slice, drill-down y dice sobre DuckDB

In [ ]:
rollup = con.execute('''
    SELECT d.anio, d.trimestre, CAST(SUM(GREATEST(f.casos_nuevos, 0)) AS BIGINT) AS casos_nuevos
    FROM fact_covid f JOIN dim_fecha d ON f.fecha_key = d.fecha_key
    GROUP BY 1, 2 ORDER BY 1, 2
''').df()
display(rollup.head(8))

dice_cl = con.execute('''
    SELECT t.province_state AS region, t.population,
           CAST(SUM(GREATEST(f.casos_nuevos, 0)) AS BIGINT) AS casos_inv_2021,
           ROUND(SUM(GREATEST(f.casos_nuevos, 0)) * 100000.0 / NULLIF(t.population, 0), 1) AS tasa_100k
    FROM fact_covid f
    JOIN dim_territorio t ON f.territorio_key = t.territorio_key
    JOIN dim_fecha d ON f.fecha_key = d.fecha_key
    WHERE t.country_region = 'Chile' AND t.province_state IS NOT NULL AND t.province_state <> 'Unknown'
      AND d.anio = 2021 AND d.mes IN (6, 7, 8)
    GROUP BY 1, 2 ORDER BY tasa_100k DESC
''').df()
display(dice_cl)
con.close()